# 03. Variant-aware sequence database

Builds the FASTA file used for the re-search:

| Part | Source | Purpose |
|---|---|---|
| Reference proteome | UniProtKB/Swiss-Prot, human, canonical sequences | standard identification |
| Variant peptides | UniProt `humsavar` (all curated single amino-acid variants) | detect donor-specific variant peptides |
| Contaminants | universal contaminant library (Frankenfield et al. 2022, J Proteome Res) | keratins, trypsin, serum proteins |
| Decoys | reversed sequences, prefix `rev_` | false discovery rate |

A variant is not added as a full-length protein. Only the tryptic neighbourhood of the substituted residue is added (the peptide carrying the variant plus up to two cleavage sites on each side), which keeps the added sequence small (about 29% more residues, most of them flanking peptides identical to the reference).

Three groups of variants are tracked for the analysis (notebook 06):

1. **all** curated variants, proteome-wide (the method's positive control);
2. **panel**: cytochromes P450 detected in brain, their redox partners, conjugating enzymes and genes of the four MOSAIC-S pathways;
3. **pharmacogenes**: CYP2D6, CYP2C19, CYP2C9, CYP2B6, CYP3A4/5, expected to be near or below the detection limit in brain.

UniProt release used for the published run: **2026_04** (4 November 2026).

In [1]:
from pathlib import Path
import re, urllib.request, datetime, hashlib
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
EXT = ROOT / "data_external"; EXT.mkdir(exist_ok=True)
DB = ROOT / "database"; DB.mkdir(exist_ok=True)
RES = ROOT / "results"; RES.mkdir(exist_ok=True)

SOURCES = {
    "uniprot_human_reviewed.fasta": "https://rest.uniprot.org/uniprotkb/stream?query=organism_id:9606+AND+reviewed:true&format=fasta",
    "humsavar.txt": "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/variants/humsavar.txt",
    "contaminants_HaoLab_universal.fasta": "https://raw.githubusercontent.com/HaoGroup-ProtContLib/Protein-Contaminant-Libraries-for-DDA-and-DIA-Proteomics/main/Universal%20protein%20contaminant%20FASTA/0602_Universal%20Contaminants.fasta",
}
for name, url in SOURCES.items():
    if not (EXT / name).exists():
        urllib.request.urlretrieve(url, EXT / name)
    print(name, round((EXT / name).stat().st_size / 1e6, 1), "MB")
release = next(l for l in open(EXT / "humsavar.txt", encoding="utf-8") if l.startswith("Release:")).strip()
print(release)

# The results in this repository were produced with UniProt release 2026_04 (Swiss-Prot FASTA and humsavar, 4 November 2026).
# UniProt serves only the current release at these addresses; a later download gives a slightly different database.
# The exact FASTA used for the search is identified by its SHA-1 in results/database_summary.tsv and archived with the release of this repository.
EXPECTED_RELEASE = "2026_04"
if EXPECTED_RELEASE not in release:
    print(f"NOTE: downloaded release differs from {EXPECTED_RELEASE}; variant counts and the FASTA checksum will differ from the published run.")

uniprot_human_reviewed.fasta 13.7 MB
humsavar.txt 8.7 MB
contaminants_HaoLab_universal.fasta 0.2 MB
Release:     2026_04 of 04-Nov-2026


In [2]:
def read_fasta(path):
    out, head, seq = [], None, []
    for line in open(path, encoding="utf-8"):
        line = line.rstrip()
        if line.startswith(">"):
            if head: out.append((head, "".join(seq)))
            head, seq = line[1:], []
        elif line:
            seq.append(line)
    if head: out.append((head, "".join(seq)))
    return out

ref = read_fasta(EXT / "uniprot_human_reviewed.fasta")
cont = read_fasta(EXT / "contaminants_HaoLab_universal.fasta")
seq_of = {h.split("|")[1]: s for h, s in ref}
gene_of = {h.split("|")[1]: (re.search(r"GN=(\S+)", h).group(1) if "GN=" in h else "") for h, s in ref}
print(len(ref), "reference proteins;", len(cont), "contaminants")

20429 reference proteins; 381 contaminants


## Variants

In [3]:
AA3 = dict(Ala="A", Arg="R", Asn="N", Asp="D", Cys="C", Gln="Q", Glu="E", Gly="G", His="H", Ile="I", Leu="L", Lys="K",
           Met="M", Phe="F", Pro="P", Ser="S", Thr="T", Trp="W", Tyr="Y", Val="V", Sec="U")
rows, started = [], False
for line in open(EXT / "humsavar.txt", encoding="utf-8"):
    if line.startswith("_________"):
        started = True; continue
    if not started or not line.strip() or line.startswith("---"):
        continue
    p = line.split()
    if len(p) < 6 or not p[2].startswith("VAR_"):
        continue
    m = re.fullmatch(r"p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", p[3])
    if m and m.group(1) in AA3 and m.group(3) in AA3:
        rows.append((p[0], p[1], p[2], AA3[m.group(1)], int(m.group(2)), AA3[m.group(3)], p[4], p[5]))
var = pd.DataFrame(rows, columns=["gene", "accession", "ftid", "ref", "pos", "alt", "category", "dbsnp"]).drop_duplicates("ftid")
# keep variants whose reference residue agrees with the canonical sequence used here
ok = [seq_of.get(a, "")[p - 1:p] == r for a, p, r in zip(var["accession"], var["pos"], var["ref"])]
print(len(var), "single amino-acid variants;", sum(ok), "consistent with the reference sequences")
var = var[ok].reset_index(drop=True)
var["category"].value_counts()

84003 single amino-acid variants; 84003 consistent with the reference sequences


category
LB/B    39954
LP/P    32347
US      11702
Name: count, dtype: int64

In [4]:
PANEL = """CYP46A1 CYP27A1 CYP51A1 CYP7B1 CYP39A1 CYP2J2 CYP4X1 CYP20A1 CYP2U1 CYP1B1 CYP2E1 CYP19A1 CYP11A1 CYP1A1 CYP4F2
POR FDXR FDX1 CYB5A CYB5R3 GSTP1 GSTM1 GSTM2 GSTM3 GSTT1 GSTT2 GSTO1 COMT EPHX1 EPHX2 ALDH2 ADH1B MAOA MAOB NQO1 SULT1A1
HMGCR DHCR7 DHCR24 APOE LDLR NR3C1 NR3C2 FKBP5 HSD11B1 CRHR1 SLC6A4 TPH2 BDNF NTRK2 S100A10 GRM5 HOMER1 SLC17A7 GFAP""".split()
PHARMACOGENES = "CYP2D6 CYP2C19 CYP2C9 CYP2B6 CYP3A4 CYP3A5 CYP1A2 CYP2C8 CYP2A6".split()
var["group"] = "all"
var.loc[var["gene"].isin(PANEL), "group"] = "panel"
var.loc[var["gene"].isin(PHARMACOGENES), "group"] = "pharmacogene"
var["group"].value_counts()

group
all             83086
panel             721
pharmacogene      196
Name: count, dtype: int64

In [5]:
def window(seq, pos, alt, n_sites=2):
    """Variant sequence from the n-th cleavage site upstream to the n-th downstream of the substituted residue."""
    mut = seq[:pos - 1] + alt + seq[pos:]
    cut = [0] + [i + 1 for i, a in enumerate(mut) if a in "KR"] + [len(mut)]       # positions after K/R
    cut = sorted(set(cut))
    left = [c for c in cut if c <= pos - 1]
    right = [c for c in cut if c >= pos]
    a = left[-n_sites] if len(left) >= n_sites else 0
    b = right[n_sites - 1] if len(right) >= n_sites else len(mut)
    return mut[a:b], a

def variant_peptide(seq, pos, alt):
    """The fully tryptic peptide that carries the variant residue (used later to recognise variant evidence)."""
    w, a = window(seq, pos, alt, n_sites=1)
    return w

entries, pep = [], []
for r in var.itertuples():
    w, a = window(seq_of[r.accession], r.pos, r.alt)
    tag = f"{r.accession}_{r.ref}{r.pos}{r.alt}"
    entries.append((f"sp|{tag}|{r.gene}_{r.ref}{r.pos}{r.alt}_HUMAN Variant {r.ftid} {r.dbsnp} window {a + 1}-{a + len(w)} "
                    f"OS=Homo sapiens OX=9606 GN={r.gene} PE=1 SV=1", w))
    pep.append(variant_peptide(seq_of[r.accession], r.pos, r.alt))
var["variant_entry"] = [e[0].split("|")[1] for e in entries]
var["variant_tryptic_peptide"] = pep
var.to_csv(RES / "variant_table.tsv", sep="\t", index=False)
print(len(entries), "variant entries; median window length", int(pd.Series([len(e[1]) for e in entries]).median()))
var[var["dbsnp"].isin(["rs671", "rs429358", "rs7412", "rs4680", "rs1695", "rs1065852"])][["gene", "ref", "pos", "alt", "dbsnp", "category", "group", "variant_tryptic_peptide"]]

84003 variant entries; median window length 34


,gene,ref,pos,alt,dbsnp,category,group,variant_tryptic_peptide
3892,ALDH2,E,504,K,rs671,LP/P,panel,ELGEYGLQAYTK
5374,APOE,C,130,R,rs429358,LP/P,panel,LGADMEDVR
5385,APOE,R,176,C,rs7412,LP/P,panel,CLAVYQAGAR
16673,COMT,V,158,M,rs4680,LB/B,panel,MVDFAGMK
18808,CYP2D6,P,34,S,rs1065852,LB/B,pharmacogene,YSPGPLPLPGLGNLLHVDFQNTPYCFDQLR
32232,GSTP1,I,105,V,rs1695,LB/B,panel,YVSLIYTNYEAGK


The table above lists six well-known variants as a sanity check: *ALDH2\*2* (rs671), the two *APOE* variants defining ε2/ε4, *COMT* Val158Met, *GSTP1* Ile105Val and *CYP2D6* Pro34Ser.

## Write the database

In [6]:
def wrap(s, n=60):
    return "\n".join(s[i:i + n] for i in range(0, len(s), n))

targets = ref + entries + cont
stamp = datetime.date.today().isoformat()
out = DB / "MOSAIC_human_sp_variants_contam_decoys.fasta"
with open(out, "w", encoding="utf-8", newline="\n") as fh:
    for h, s in targets:
        fh.write(f">{h}\n{wrap(s)}\n")
    for h, s in targets:
        fh.write(f">rev_{h}\n{wrap(s[::-1])}\n")
summary = pd.Series({
    "date": stamp, "humsavar": release,
    "reference_proteins": len(ref), "variant_entries": len(entries), "contaminants": len(cont),
    "decoys": len(targets), "total_entries": 2 * len(targets),
    "residues_reference": sum(len(s) for _, s in ref), "residues_variants": sum(len(s) for _, s in entries),
    "fasta_sha1": hashlib.sha1(out.read_bytes()).hexdigest(), "fasta_mb": round(out.stat().st_size / 1e6, 1),
})
summary.to_csv(RES / "database_summary.tsv", sep="\t", header=False)
print("search space added by variants: %.1f%%" % (100 * summary["residues_variants"] / summary["residues_reference"]))
summary

search space added by variants: 28.8%


date                                                2026-11-25
humsavar                   Release:     2026_04 of 04-Nov-2026
reference_proteins                                       20429
variant_entries                                          84003
contaminants                                               381
decoys                                                  104813
total_entries                                           209626
residues_reference                                    11417710
residues_variants                                      3282668
fasta_sha1            d08575b416f1ffbe195823fe136ee6e2a2382d88
fasta_mb                                                  55.3
dtype: object